In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline 
import pandas as pd

import scipy.signal

from astropy.io import ascii
import exoplanet as xo

import pymc3 as pm
import pymc3_ext as pmx
import lightkurve as lk


In [ ]:
#Curva = ascii.read('Data_Transit_TIC427393298_Full.dat')

#Times = Curva.columns[0].astype(np.float64)
#Flux = Curva.columns[1].astype(np.float64)
curva = pd.read_csv("/home/luciam/Documents/starry/pleyades_5/TIC_346541355/Tic_346541355_s_44_corrected.csv")
Times = curva["time"]
Flux = curva["flux"]

fig = plt.figure(figsize=(12,5), dpi=300)
plt.plot(Times, Flux, ".k")
plt.ylabel("Flujo")
plt.xlabel("Tiempo")
plt.rc('font', size=16)

## De-trending usando wotan

In [ ]:
from wotan import flatten
flatten_lc, trend_lc = flatten(Times, Flux, window_length=0.80, break_tolerance=2.0, method='hspline', return_trend=True)
#, method='biweight'
#window lenght es el porcentaje de puntos que se usan para hacer la ventana de observacion
fig = plt.figure(figsize=(12,5), dpi=300)
plt.plot(Times, Flux, ".k")
plt.plot(Times, trend_lc, c='b')
plt.ylabel("Flujo")
plt.xlabel("Tiempo")
plt.rc('font', size=12)

In [ ]:
data_trend = pd.DataFrame({"Time": Times, "Flux": trend_lc})
data_trend.to_csv("tic_346541355_trend_wotan.csv", index = False)

In [ ]:
Fl_Det = Flux/trend_lc

fig = plt.figure(figsize=(12,5), dpi=300)
#plt.plot(Time, Flux, ".k")
#plt.plot(Times, Fl_Det, c='b')
plt.scatter(Times, Fl_Det, c='b',s = 10)#s tamaño de punto 
plt.ylabel("Flujo")
plt.xlabel("Tiempo")
plt.rc('font', size=12)

In [ ]:
Report = open("Data_Norm_TIC346541355_44.dat", "w")
print("#{0:9s} {1:9s}".format('Time','Flux'), file=Report) # Writing header to file
for j in range(len(Times)):
    print("{0:>9.5f} {1:>9.8f}".format(Times[j],Fl_Det[j]), file=Report)
print("# End of file",file=Report)
Report.close()

## De-trending con lightkurve

In [ ]:
#cargamos la curva de luz, porque no queremos que la descargue
lc = lk.LightCurve(time=Times, flux = Flux)
normalized_lc = lc.normalize()
flat_lc, trend_lc = normalized_lc.flatten(window_length = 400, polyorder = 2, return_trend = True)
#window lenght es el numero de puntos usado para la ventana de observaciones
fig = plt.figure(figsize=(12,5), dpi=300)

plt.plot(Times, Flux, ".k")
plt.plot(Times, trend_lc.flux, c='b')
plt.ylabel("Flujo")
plt.xlabel("Tiempo")
plt.rc('font', size=12)

In [ ]:
data_trend = pd.DataFrame({"Time": Times, "Flux": trend_lc.flux})
data_trend.to_csv("TIC_346541355_trend_lk.csv", index = False)

In [ ]:
Fl_Det = Flux/trend_lc.flux

fig = plt.figure(figsize=(12,5), dpi=300)
#plt.plot(Time, Flux, ".k")
#plt.plot(Times, Fl_Det, c='b')
plt.scatter(Times, Fl_Det, c='b',s = 10)#s tamaño de punto 
plt.ylabel("Flujo")
plt.xlabel("Tiempo")
plt.rc('font', size=12)